# Faster R-CNN Bone Fracture Detection - Model Training

This notebook trains a **Faster R-CNN** model using the dataset prepared by the **R-CNN data preparation notebook**.

## Expected input
This notebook expects the prepared dataset folder to already exist, usually at:

`/content/bone_fracture_rcnn_ready`

with this structure:

- `train/images`
- `valid/images`
- `test/images`
- `annotations/instances_train.json`
- `annotations/instances_valid.json`
- `annotations/instances_test.json`

## What this notebook does
1. Installs required packages
2. Loads the prepared COCO-style dataset
3. Builds a Faster R-CNN model
4. Trains the model
5. Tracks training and validation loss
6. Saves the best model weights
7. Runs inference on a few test images

In [1]:
# Install required packages
!pip install -q torch torchvision pycocotools matplotlib pillow

In [2]:
import os
import json
import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

import torch
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.transforms import functional as F
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

# Check device
print("Device Check:")
print(f"  CUDA available: {torch.cuda.is_available()}")
print(f"  MPS available: {torch.backends.mps.is_available() if hasattr(torch.backends, 'mps') else False}")
print("  CPU: Always available")

if torch.cuda.is_available():
    device = torch.device("cuda")
    device_name = "CUDA GPU"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
    device_name = "Apple MPS"
else:
    device = torch.device("cpu")
    device_name = "CPU"

print(f"\nUsing device: {device_name} ({device})")

Device Check:
  CUDA available: False
  MPS available: False
  CPU: Always available

Using device: CPU (cpu)


In [11]:
import os

DATA_ROOT = "/content/bone_fracture_rcnn_ready"

print("Exists:", os.path.exists(DATA_ROOT))
print("Folders:", os.listdir(DATA_ROOT) if os.path.exists(DATA_ROOT) else "Not found")

Exists: False
Folders: Not found


## Paths and training configuration

Update these values if your prepared dataset is saved somewhere else.

In [10]:
# Dataset paths
dataset_root = Path("/content/bone_fracture_rcnn_ready")
train_images_dir = dataset_root / "train" / "images"
valid_images_dir = dataset_root / "valid" / "images"
test_images_dir  = dataset_root / "test" / "images"

train_ann_path = dataset_root / "annotations" / "instances_train.json"
valid_ann_path = dataset_root / "annotations" / "instances_valid.json"
test_ann_path  = dataset_root / "annotations" / "instances_test.json"

# Output paths
output_dir = Path("/content/runs/faster_rcnn_bone_fracture")
output_dir.mkdir(parents=True, exist_ok=True)

best_model_path = output_dir / "best_faster_rcnn.pth"
last_model_path = output_dir / "last_faster_rcnn.pth"

# Training hyperparameters
NUM_EPOCHS = 15
BATCH_SIZE = 4 if device.type != "cpu" else 2
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2 if device.type != "cpu" else 0
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

required_paths = [
    train_images_dir, valid_images_dir, test_images_dir,
    train_ann_path, valid_ann_path, test_ann_path
]

missing = [str(p) for p in required_paths if not p.exists()]
if missing:
    raise FileNotFoundError(
        "These required dataset files/folders were not found:\n" + "\n".join(missing)
    )

print("Dataset root:", dataset_root)
print("Output dir:", output_dir)
print("Train annotations:", train_ann_path)
print("Valid annotations:", valid_ann_path)
print("Test annotations:", test_ann_path)

FileNotFoundError: These required dataset files/folders were not found:
/content/bone_fracture_rcnn_ready/train/images
/content/bone_fracture_rcnn_ready/valid/images
/content/bone_fracture_rcnn_ready/test/images
/content/bone_fracture_rcnn_ready/annotations/instances_train.json
/content/bone_fracture_rcnn_ready/annotations/instances_valid.json
/content/bone_fracture_rcnn_ready/annotations/instances_test.json

## Load class information from the COCO annotations

In [ ]:
with open(train_ann_path, "r") as f:
    train_coco = json.load(f)

categories = sorted(train_coco["categories"], key=lambda x: x["id"])
category_id_to_name = {c["id"]: c["name"] for c in categories}
category_name_to_id = {c["name"]: c["id"] for c in categories}

# Faster R-CNN expects class indices from 1..N, with 0 reserved for background.
# Our prepared COCO file should already use positive category ids.
num_classes = len(categories) + 1

print("Categories:")
for c in categories:
    print(f"  {c['id']}: {c['name']}")
print(f"\nNumber of object classes: {len(categories)}")
print(f"Number of classes passed to model (including background): {num_classes}")

## Dataset class for COCO-style annotations
This reads the prepared annotation JSON and returns images with bounding boxes and labels in the format expected by Faster R-CNN.

In [ ]:
class BoneFractureCocoDataset(Dataset):
    def __init__(self, images_dir, annotation_path):
        self.images_dir = Path(images_dir)
        self.annotation_path = Path(annotation_path)

        with open(self.annotation_path, "r") as f:
            coco = json.load(f)

        self.images = sorted(coco["images"], key=lambda x: x["id"])
        self.categories = coco["categories"]
        self.annotations = coco["annotations"]

        self.image_id_to_info = {img["id"]: img for img in self.images}
        self.image_id_to_anns = {}
        for ann in self.annotations:
            self.image_id_to_anns.setdefault(ann["image_id"], []).append(ann)

        self.ids = [img["id"] for img in self.images]

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        image_id = self.ids[idx]
        image_info = self.image_id_to_info[image_id]
        image_path = self.images_dir / image_info["file_name"]

        image = Image.open(image_path).convert("RGB")
        anns = self.image_id_to_anns.get(image_id, [])

        boxes = []
        labels = []
        areas = []
        iscrowd = []

        for ann in anns:
            x, y, w, h = ann["bbox"]
            x2 = x + w
            y2 = y + h

            # skip invalid boxes
            if w <= 0 or h <= 0:
                continue
            if x2 <= x or y2 <= y:
                continue

            boxes.append([x, y, x2, y2])
            labels.append(int(ann["category_id"]))
            areas.append(float(ann.get("area", w * h)))
            iscrowd.append(int(ann.get("iscrowd", 0)))

        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)
        areas = torch.as_tensor(areas, dtype=torch.float32)
        iscrowd = torch.as_tensor(iscrowd, dtype=torch.int64)

        if boxes.numel() == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
            areas = torch.zeros((0,), dtype=torch.float32)
            iscrowd = torch.zeros((0,), dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([image_id], dtype=torch.int64),
            "area": areas,
            "iscrowd": iscrowd,
        }

        image = F.to_tensor(image)
        return image, target


def collate_fn(batch):
    return tuple(zip(*batch))

In [ ]:
train_dataset = BoneFractureCocoDataset(train_images_dir, train_ann_path)
valid_dataset = BoneFractureCocoDataset(valid_images_dir, valid_ann_path)
test_dataset  = BoneFractureCocoDataset(test_images_dir, test_ann_path)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
)

print(f"Train images: {len(train_dataset)}")
print(f"Valid images: {len(valid_dataset)}")
print(f"Test images:  {len(test_dataset)}")

## Quick sanity check

In [ ]:
sample_img, sample_target = train_dataset[0]
print("Image tensor shape:", sample_img.shape)
print("Target keys:", sample_target.keys())
print("Boxes shape:", sample_target["boxes"].shape)
print("Labels:", sample_target["labels"][:10])

## Build the Faster R-CNN model
This starts from pretrained COCO backbone weights and replaces the final classifier head for the fracture classes.

In [ ]:
def get_model(num_classes):
    weights = torchvision.models.detection.FasterRCNN_ResNet50_FPN_Weights.DEFAULT
    model = fasterrcnn_resnet50_fpn(weights=weights)

    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

model = get_model(num_classes)
model.to(device)
print("Model initialized: Faster R-CNN (ResNet-50 FPN)")

In [ ]:
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(params, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

## Training and validation functions
Validation here is based on detection loss on the validation split. This keeps the notebook simple and reliable for Colab.

In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0

    for images, targets in loader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        loss = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / max(len(loader), 1)


@torch.no_grad()
def validate_one_epoch(model, loader, device):
    # torchvision detection models only return losses in train mode when targets are provided.
    model.train()
    total_loss = 0.0

    for images, targets in loader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        loss = sum(loss for loss in loss_dict.values())
        total_loss += loss.item()

    return total_loss / max(len(loader), 1)

## Train the model
Best weights are saved using validation loss.

In [ ]:
history = {
    "train_loss": [],
    "valid_loss": [],
}

best_val_loss = float("inf")

for epoch in range(NUM_EPOCHS):
    train_loss = train_one_epoch(model, train_loader, optimizer, device)
    valid_loss = validate_one_epoch(model, valid_loader, device)

    lr_scheduler.step()

    history["train_loss"].append(train_loss)
    history["valid_loss"].append(valid_loss)

    print(
        f"Epoch [{epoch + 1}/{NUM_EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Valid Loss: {valid_loss:.4f}"
    )

    torch.save(
        {
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "train_loss": train_loss,
            "valid_loss": valid_loss,
            "categories": categories,
        },
        last_model_path,
    )

    if valid_loss < best_val_loss:
        best_val_loss = valid_loss
        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "train_loss": train_loss,
                "valid_loss": valid_loss,
                "categories": categories,
            },
            best_model_path,
        )
        print(f"  Saved new best model to: {best_model_path}")

print("\nTraining finished.")
print("Best validation loss:", best_val_loss)

## Plot training history

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history["train_loss"], label="Train Loss")
plt.plot(history["valid_loss"], label="Valid Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Faster R-CNN Training History")
plt.legend()
plt.grid(True)
plt.show()

## Load the best model

In [ ]:
checkpoint = torch.load(best_model_path, map_location=device)
model = get_model(num_classes)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)
model.eval()

print(f"Loaded best model from: {best_model_path}")
print(f"Best checkpoint epoch: {checkpoint['epoch']}")
print(f"Validation loss at save: {checkpoint['valid_loss']:.4f}")

## Run inference on a few test images
This shows predicted boxes and class names on test images.

In [ ]:
def draw_predictions(image_pil, boxes, labels, scores, score_thresh=0.3):
    image_draw = image_pil.copy()
    draw = ImageDraw.Draw(image_draw)

    for box, label, score in zip(boxes, labels, scores):
        if score < score_thresh:
            continue

        x1, y1, x2, y2 = [float(v) for v in box]
        class_name = category_id_to_name.get(int(label), f"class_{int(label)}")
        text = f"{class_name}: {score:.2f}"

        draw.rectangle([x1, y1, x2, y2], outline="red", width=3)
        draw.text((x1, max(0, y1 - 12)), text, fill="red")

    return image_draw


num_examples = min(5, len(test_dataset))
score_threshold = 0.3

for idx in range(num_examples):
    image_tensor, target = test_dataset[idx]
    image_pil = F.to_pil_image(image_tensor)

    with torch.no_grad():
        predictions = model([image_tensor.to(device)])[0]

    pred_boxes = predictions["boxes"].detach().cpu()
    pred_labels = predictions["labels"].detach().cpu()
    pred_scores = predictions["scores"].detach().cpu()

    vis = draw_predictions(
        image_pil=image_pil,
        boxes=pred_boxes,
        labels=pred_labels,
        scores=pred_scores,
        score_thresh=score_threshold,
    )

    plt.figure(figsize=(8, 8))
    plt.imshow(vis)
    plt.title(f"Test Image {idx} Predictions")
    plt.axis("off")
    plt.show()

## Optional: save one inference image to disk

In [ ]:
save_example_path = output_dir / "sample_prediction.png"

image_tensor, _ = test_dataset[0]
image_pil = F.to_pil_image(image_tensor)

with torch.no_grad():
    predictions = model([image_tensor.to(device)])[0]

vis = draw_predictions(
    image_pil=image_pil,
    boxes=predictions["boxes"].detach().cpu(),
    labels=predictions["labels"].detach().cpu(),
    scores=predictions["scores"].detach().cpu(),
    score_thresh=0.3,
)

vis.save(save_example_path)
print("Saved sample prediction to:", save_example_path)